# Music Annotation Project V7.7: Asymmetric Structural-Risk Calibration

**Runtime:** CPU. Select **Runtime > Run all**.

V7.7 freezes the completed V7.5 `geometry_t80` reconstruction and replaces V7.6's single symmetric `safe/review_required` target with two separate post-hoc outcomes:

1. **Structural safety**: did the correction preserve genuine printed musical information?
2. **Cosmetic quality**: did annotation or tonal residue remain even when musical information was preserved?

The primary model predicts **structural safety**. Cosmetic residue is reported and modelled separately. Clean truth is used only after reconstruction to construct these outcomes and is excluded from predictors.

## Hard guardrails

- V7.5 reconstruction is not rerun or changed.
- Only `geometry_t80` components are used.
- Pixels outside the controlled removal mask remain outside this calibration problem.
- Cross-validation is grouped by page.
- Raw page coordinates `x` and `y` are never predictors.
- Clean/truth/outcome variables are never predictors.
- Candidate thresholds are retrospective calibration findings, not production authorisation.
- Independent score/source-group validation remains mandatory before auto-acceptance.

## Primary label

A component is `structural_safe` only when all structural gates pass. Residual annotation does **not** make a component structurally unsafe. It is instead assigned a separate cosmetic-residual score and label.

Default gates are conservative proposals and are exported in the manifest. Sensitivity tables test alternative structural and cosmetic definitions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib scikit-learn


In [ ]:
import cv2, json, re, math, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve, confusion_matrix
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V76_ROOT=ROOT/'correction_safety_calibration_v7_6'
V75_RUN_OVERRIDE=None
V76_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'structural_risk_calibration_v7_7'
RANDOM_SEED=42
MAX_CONTACTS=40

STRUCTURAL_GATES={
 'minimum_ink_recall':0.985,
 'minimum_ink_f1':0.95,
 'minimum_edge_f1':0.90,
 'maximum_removed_print_fraction':0.001,
 'maximum_component_mae':16.0,
}
COSMETIC_GATES={
 'maximum_residual_annotation_fraction':0.02,
 'maximum_background_mae':12.0,
}

def complete_v75(p):
 req=[p/'benchmark_manifest.json',p/'tables'/'surrounding_area_evidence_by_component.csv',p/'tables'/'surrounding_evidence_truth_diagnostics.csv',p/'tables'/'cache_status.csv']
 return all(x.exists() for x in req)
def complete_v76(p): return (p/'v7_6_manifest.json').exists() and (p/'tables'/'calibration_dataset.csv').exists()
def newest(root,check):
 candidates=[p for p in (root/'runs').iterdir() if p.is_dir() and check(p)] if (root/'runs').exists() else []
 if not candidates: candidates=[p for p in root.iterdir() if p.is_dir() and check(p)] if root.exists() else []
 return max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None
V75_RUN=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else newest(V75_ROOT,complete_v75)
V76_RUN=Path(V76_RUN_OVERRIDE) if V76_RUN_OVERRIDE else newest(V76_ROOT,complete_v76)
if V75_RUN is None: raise RuntimeError('No complete V7.5 run found. Set V75_RUN_OVERRIDE.')
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables';CHARTS=OUT/'charts';CONTACTS=OUT/'contact_sheets'
for d in [OUT,TABLES,CHARTS,CONTACTS]:d.mkdir(parents=True,exist_ok=True)
print('V7.5:',V75_RUN);print('V7.6 reference:',V76_RUN);print('V7.7:',OUT)


## Load frozen V7.5 evidence and post-hoc truth diagnostics


In [ ]:
def read_required(path):
 if not path.exists():raise RuntimeError(f'Missing required file: {path}')
 return pd.read_csv(path)
def select_t80(frame):
 for c in ['method','candidate','geometry_candidate']:
  if c in frame.columns:return frame[frame[c].astype(str).str.lower().str.contains('t80')].copy()
 raise RuntimeError('No candidate field available to select geometry_t80.')
evidence=select_t80(read_required(V75_RUN/'tables'/'surrounding_area_evidence_by_component.csv'))
truth=select_t80(read_required(V75_RUN/'tables'/'surrounding_evidence_truth_diagnostics.csv'))
keys=[c for c in ['page','component','method','candidate','geometry_candidate'] if c in evidence.columns and c in truth.columns]
if not {'page','component'}.issubset(keys):raise RuntimeError('V7.5 evidence and truth tables must share page/component.')
df=evidence.merge(truth[keys+[c for c in truth.columns if c not in keys]],on=keys,how='inner',validate='one_to_one',suffixes=('','_truth'))
print('Matched components:',len(df),'pages:',df.page.nunique())


## Recover frozen `geometry_t80` component outcomes from V7.5 caches


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def tolerant_scores(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.sum():return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8)
 precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
 recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
 return float(precision),float(recall),float(2*precision*recall/max(precision+recall,1e-12))
clean_map=image_map(ROOT/'source'/'clean')
cache_status=read_required(V75_RUN/'tables'/'cache_status.csv')
cache_lookup={str(r.page):Path(r.cache) for _,r in cache_status.iterrows()}
rows=[]
for page,g in df.groupby('page'):
 page=str(page);cp=clean_map.get(page);zp=cache_lookup.get(page)
 if cp is None or zp is None or not zp.exists():continue
 clean=cv2.imread(str(cp),0);z=np.load(zp);mask=z['mask']>0;recon=z['geometry_t80'];n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
 clean_ink=printed_ink(clean);recon_ink=printed_ink(recon);clean_edges=edges(clean);recon_edges=edges(recon)
 for _,r in g.iterrows():
  cid=int(r.component)
  if cid<=0 or cid>=n:continue
  comp=lab==cid;ip,ir,if1=tolerant_scores(clean_ink,recon_ink,comp);ep,er,ef1=tolerant_scores(clean_edges,recon_edges,comp)
  diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32))
  background=comp&~clean_ink
  rows.append({'page':page,'component':cid,'component_pixels':int(comp.sum()),'component_mae':float(diff[comp].mean()),'background_mae':float(diff[background].mean()) if background.any() else np.nan,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'x_cache':int(stats[cid,0]),'y_cache':int(stats[cid,1]),'w_cache':int(stats[cid,2]),'h_cache':int(stats[cid,3])})
outcomes=pd.DataFrame(rows);df=df.merge(outcomes,on=['page','component'],how='inner',validate='one_to_one')
print('Scored components:',len(df),'pages:',df.page.nunique())


## Build asymmetric structural and cosmetic targets


In [ ]:
def numeric_alias(frame,names,default=np.nan):
 for n in names:
  if n in frame.columns:return pd.to_numeric(frame[n],errors='coerce')
 return pd.Series(default,index=frame.index,dtype=float)
removed_print=numeric_alias(df,['removed_clean_print_fraction','remove_clean_print_fraction','removed_print_fraction'],0.0).fillna(0)
residual_annotation=(1-numeric_alias(df,['preserved_clean_print_precision','preserve_clean_print_precision'],1.0)).clip(0,1).fillna(0)
# Structural hard failure: any gate indicates material risk to genuine printed notation.
structural_checks={
 'ink_recall':df.ink_recall.fillna(1)>=STRUCTURAL_GATES['minimum_ink_recall'],
 'ink_f1':df.ink_f1.fillna(1)>=STRUCTURAL_GATES['minimum_ink_f1'],
 'edge_f1':df.edge_f1.fillna(1)>=STRUCTURAL_GATES['minimum_edge_f1'],
 'removed_print':removed_print<=STRUCTURAL_GATES['maximum_removed_print_fraction'],
 'component_mae':df.component_mae<=STRUCTURAL_GATES['maximum_component_mae'],
}
df['structural_safe']=np.logical_and.reduce(list(structural_checks.values()))
df['structural_outcome']=np.where(df.structural_safe,'structural_safe','structural_unsafe')
df['structural_failure_reasons']=''
for name,passed in structural_checks.items():df.loc[~passed,'structural_failure_reasons']=df.loc[~passed,'structural_failure_reasons'].map(lambda x:(x+'|'+name).strip('|'))
df['residual_annotation_fraction']=residual_annotation
df['cosmetic_clean']=(residual_annotation<=COSMETIC_GATES['maximum_residual_annotation_fraction'])&(df.background_mae.fillna(0)<=COSMETIC_GATES['maximum_background_mae'])
df['cosmetic_outcome']=np.where(df.cosmetic_clean,'cosmetic_clean','cosmetic_residual')
df['four_way_outcome']=np.select([~df.structural_safe,df.structural_safe&~df.cosmetic_clean],[ 'structural_unsafe','structural_safe_cosmetic_residual'],default='structural_safe_cosmetic_clean')
print(df.structural_outcome.value_counts());print(df.cosmetic_outcome.value_counts());print(df.four_way_outcome.value_counts())


## Predictor firewall


In [ ]:
# Clean truth, outcome measures, page identity and raw coordinates are forbidden predictors.
IDENTIFIERS={'page','component','method','candidate','geometry_candidate'}
EXACT_EXCLUSIONS={'x','y','x_cache','y_cache','w_cache','h_cache'}
TRUTH_PATTERNS=['clean','truth','precision','recall','component_mae','background_mae','ink_f1','edge_f1','structural','cosmetic','residual_annotation','removed_print','outcome','failure']
numeric_cols=list(df.select_dtypes(include=[np.number,bool]).columns)
predictors=[]
for c in numeric_cols:
 if c in IDENTIFIERS or c in EXACT_EXCLUSIONS:continue
 if any(p in c.lower() for p in TRUTH_PATTERNS):continue
 predictors.append(c)
X=df[predictors].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
for c in list(X):
 finite=X[c].dropna()
 if finite.empty or finite.nunique()<=1 or (finite.abs()>1e9).mean()>0.01:X=X.drop(columns=c)
predictors=list(X.columns)
if not predictors:raise RuntimeError('No eligible pre-truth predictors remain.')
assert not {'x','y'}.intersection(predictors)
assert not any(any(p in c.lower() for p in TRUTH_PATTERNS) for c in predictors)
print('Eligible predictors:',predictors)
df.to_csv(TABLES/'asymmetric_component_dataset.csv',index=False)
pd.DataFrame({'predictor':predictors}).to_csv(TABLES/'predictor_allowlist.csv',index=False)


## Outcome-definition sensitivity


In [ ]:
rows=[]
for ink_recall in [.95,.975,.985,.99]:
 for ink_f1 in [.90,.95,.98]:
  for edge_f1 in [.85,.90,.95]:
   structural=(df.ink_recall.fillna(1)>=ink_recall)&(df.ink_f1.fillna(1)>=ink_f1)&(df.edge_f1.fillna(1)>=edge_f1)&(removed_print<=STRUCTURAL_GATES['maximum_removed_print_fraction'])&(df.component_mae<=STRUCTURAL_GATES['maximum_component_mae'])
   for residue in [.01,.02,.05,.10]:
    cosmetic=(residual_annotation<=residue)&(df.background_mae.fillna(0)<=COSMETIC_GATES['maximum_background_mae'])
    rows.append({'minimum_ink_recall':ink_recall,'minimum_ink_f1':ink_f1,'minimum_edge_f1':edge_f1,'maximum_residual_annotation_fraction':residue,'structural_safe':int(structural.sum()),'structural_unsafe':int((~structural).sum()),'structural_safe_fraction':float(structural.mean()),'cosmetic_clean':int(cosmetic.sum()),'cosmetic_residual':int((~cosmetic).sum()),'safe_but_cosmetic_residual':int((structural&~cosmetic).sum())})
pd.DataFrame(rows).to_csv(TABLES/'asymmetric_label_sensitivity.csv',index=False)


## Page-grouped out-of-fold modelling


In [ ]:
groups=df.page.astype(str);unique_pages=groups.nunique()
if unique_pages<3:raise RuntimeError('At least three pages are required.')
splitter=LeaveOneGroupOut() if unique_pages<=12 else GroupKFold(n_splits=min(10,unique_pages))
models={
 'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=RANDOM_SEED))]),
 'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=800,min_samples_leaf=5,max_features='sqrt',class_weight='balanced_subsample',random_state=RANDOM_SEED,n_jobs=-1))]),
 'hist_gradient':Pipeline([('impute',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,min_samples_leaf=12,l2_regularization=2.0,random_state=RANDOM_SEED))]),
}
def run_target(target_name,y):
 summaries=[];folds=[];predictions={}
 for name,model in models.items():
  oof=np.full(len(df),np.nan);fold=0
  for train,test in splitter.split(X,y,groups):
   fold+=1
   if y.iloc[train].nunique()<2:continue
   model.fit(X.iloc[train],y.iloc[train]);oof[test]=model.predict_proba(X.iloc[test])[:,1]
   valid=np.isfinite(oof[test]);yt=y.iloc[test].to_numpy()[valid];pt=oof[test][valid]
   folds.append({'target':target_name,'model':name,'fold':fold,'test_pages':'|'.join(sorted(groups.iloc[test].unique())),'components':len(test),'positive_components':int(yt.sum()),'roc_auc':roc_auc_score(yt,pt) if len(np.unique(yt))==2 else np.nan,'average_precision':average_precision_score(yt,pt) if yt.sum()>0 else np.nan,'brier':brier_score_loss(yt,pt)})
  valid=np.isfinite(oof)
  summaries.append({'target':target_name,'model':name,'roc_auc':roc_auc_score(y[valid],oof[valid]),'average_precision':average_precision_score(y[valid],oof[valid]),'brier':brier_score_loss(y[valid],oof[valid]),'components':int(valid.sum()),'pages':int(groups[valid].nunique()),'positive_prevalence':float(y[valid].mean())})
  predictions[name]=oof
 return pd.DataFrame(summaries),pd.DataFrame(folds),predictions
struct_y=df.structural_safe.astype(int);cosmetic_y=df.cosmetic_clean.astype(int)
struct_summary,struct_folds,struct_preds=run_target('structural_safe',struct_y)
cos_summary,cos_folds,cos_preds=run_target('cosmetic_clean',cosmetic_y)
cv_summary=pd.concat([struct_summary,cos_summary],ignore_index=True);fold_df=pd.concat([struct_folds,cos_folds],ignore_index=True)
cv_summary.to_csv(TABLES/'page_grouped_model_summary.csv',index=False);fold_df.to_csv(TABLES/'page_grouped_fold_results.csv',index=False)
for name,p in struct_preds.items():df[f'raw_structural_safe_score_{name}']=p
for name,p in cosmetic_preds.items():df[f'raw_cosmetic_clean_score_{name}']=p
best_struct=struct_summary.sort_values(['average_precision','roc_auc'],ascending=False).iloc[0].model
best_cosmetic=cos_summary.sort_values(['average_precision','roc_auc'],ascending=False).iloc[0].model
print('Best structural:',best_struct);display(struct_summary.sort_values('average_precision',ascending=False));print('Best cosmetic:',best_cosmetic);display(cos_summary.sort_values('average_precision',ascending=False))


## Nested page-grouped probability calibration


In [ ]:
# Calibrate only from out-of-fold predictions. Isotonic calibration maps ranking scores to empirical probabilities.
def isotonic_oof(raw,y,groups):
 calibrated=np.full(len(y),np.nan)
 outer=LeaveOneGroupOut() if groups.nunique()<=12 else GroupKFold(n_splits=min(10,groups.nunique()))
 for train,test in outer.split(raw.reshape(-1,1),y,groups):
  valid_train=np.isfinite(raw[train])
  if valid_train.sum()<20 or y.iloc[train][valid_train].nunique()<2:continue
  iso=IsotonicRegression(out_of_bounds='clip').fit(raw[train][valid_train],y.iloc[train].to_numpy()[valid_train])
  calibrated[test]=iso.predict(raw[test])
 return calibrated
raw_struct=df[f'raw_structural_safe_score_{best_struct}'].to_numpy();raw_cos=df[f'raw_cosmetic_clean_score_{best_cosmetic}'].to_numpy()
df['prob_structural_safe']=isotonic_oof(raw_struct,struct_y,groups)
df['prob_cosmetic_clean']=isotonic_oof(raw_cos,cosmetic_y,groups)
for target,prob,y in [('structural_safe','prob_structural_safe',struct_y),('cosmetic_clean','prob_cosmetic_clean',cosmetic_y)]:
 valid=df[prob].notna();print(target,'calibrated brier',brier_score_loss(y[valid],df.loc[valid,prob]))


## Candidate structural-safety policies with one-sided confidence bounds


In [ ]:
# Policies are ranked by false-safe risk. Wilson upper bounds expose uncertainty from small selected samples.
def wilson_upper(errors,n,z=1.96):
 if n==0:return np.nan
 p=errors/n;den=1+z*z/n;centre=p+z*z/(2*n);radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (centre+radius)/den
policies=[];p=df.prob_structural_safe.to_numpy();truth=df.structural_safe.to_numpy();valid=np.isfinite(p)
for threshold in np.linspace(0,1,1001):
 selected=valid&(p>=threshold);n=int(selected.sum())
 if n==0:continue
 errors=int(np.sum(selected&~truth));observed=errors/n;upper=wilson_upper(errors,n)
 policies.append({'probability_threshold':float(threshold),'selected_components':n,'selected_fraction':n/len(df),'structural_unsafe_selected':errors,'observed_false_safe_rate':observed,'false_safe_rate_wilson95_upper':upper,'structural_safe_recall':float(np.sum(selected&truth)/max(truth.sum(),1)),'pages_selected':int(df.loc[selected,'page'].nunique())})
policy_df=pd.DataFrame(policies).drop_duplicates(['selected_components','structural_unsafe_selected']).sort_values(['false_safe_rate_wilson95_upper','selected_components'],ascending=[True,False])
policy_df.to_csv(TABLES/'candidate_structural_safety_policies.csv',index=False)
# Diagnostic threshold: zero observed structural failures, maximise selected coverage. It is not production authorisation.
zero=policy_df[policy_df.structural_unsafe_selected==0]
selected_policy=zero.sort_values('selected_components',ascending=False).iloc[0] if not zero.empty else None
t=float(selected_policy.probability_threshold) if selected_policy is not None else 1.0
df['candidate_structural_auto_safe']=df.prob_structural_safe>=t
print('Diagnostic threshold:',t);display(policy_df.head(20))


## Feature evidence and stability


In [ ]:
importance=[]
for target_name,y,best in [('structural_safe',struct_y,best_struct),('cosmetic_clean',cosmetic_y,best_cosmetic)]:
 model=models[best];model.fit(X,y);q=permutation_importance(model,X,y,n_repeats=30,random_state=RANDOM_SEED,scoring='average_precision')
 for c,mean,std in zip(predictors,q.importances_mean,q.importances_std):importance.append({'target':target_name,'model':best,'predictor':c,'importance_mean':float(mean),'importance_std':float(std)})
importance_df=pd.DataFrame(importance);importance_df.to_csv(TABLES/'permutation_feature_importance.csv',index=False)
for target,g in importance_df.groupby('target'):
 q=g.sort_values('importance_mean',ascending=False).head(20).sort_values('importance_mean');plt.figure(figsize=(9,7));plt.barh(q.predictor,q.importance_mean,xerr=q.importance_std);plt.title(f'V7.7 feature importance: {target}');plt.xlabel('Decrease in average precision');plt.tight_layout();plt.savefig(CHARTS/f'feature_importance_{target}.png',dpi=170);plt.close()


## Calibration, trade-off and page-stability charts


In [ ]:
def calibration_plot(prob,y,title,path):
 valid=np.isfinite(prob);q=pd.DataFrame({'p':prob[valid],'y':y.to_numpy()[valid]});q['bin']=pd.qcut(q.p.rank(method='first'),q=min(10,len(q)),duplicates='drop');cal=q.groupby('bin',observed=True).agg(predicted=('p','mean'),observed=('y','mean'),components=('y','size'))
 plt.figure(figsize=(7,6));plt.plot(cal.predicted,cal.observed,marker='o');plt.plot([0,1],[0,1],'k--');plt.xlabel('Calibrated predicted probability');plt.ylabel('Observed fraction');plt.title(title);plt.tight_layout();plt.savefig(path,dpi=170);plt.close();return cal
struct_cal=calibration_plot(df.prob_structural_safe.to_numpy(),struct_y,'V7.7 structural-safety calibration',CHARTS/'structural_safety_calibration.png')
cos_cal=calibration_plot(df.prob_cosmetic_clean.to_numpy(),cosmetic_y,'V7.7 cosmetic-quality calibration',CHARTS/'cosmetic_quality_calibration.png')
struct_cal.to_csv(TABLES/'structural_calibration_bins.csv');cos_cal.to_csv(TABLES/'cosmetic_calibration_bins.csv')
if not policy_df.empty:
 q=policy_df.sort_values('selected_fraction');plt.figure(figsize=(8,6));plt.plot(q.selected_fraction,q.false_safe_rate_wilson95_upper);plt.xlabel('Fraction selected for candidate auto-safe tier');plt.ylabel('95% upper bound on structural false-safe rate');plt.title('Coverage versus structural-risk uncertainty');plt.tight_layout();plt.savefig(CHARTS/'coverage_vs_structural_risk.png',dpi=170);plt.close()
page=df.groupby('page').apply(lambda g:pd.Series({'components':len(g),'structural_safe':int(g.structural_safe.sum()),'candidate_auto_safe':int(g.candidate_structural_auto_safe.sum()),'false_safe':int((g.candidate_structural_auto_safe&~g.structural_safe).sum()),'safe_but_not_selected':int((~g.candidate_structural_auto_safe&g.structural_safe).sum()),'selected_cosmetic_residual':int((g.candidate_structural_auto_safe&~g.cosmetic_clean).sum())}),include_groups=False).reset_index()
page.to_csv(TABLES/'candidate_policy_by_page.csv',index=False)


## Diagnostic contact sheets


In [ ]:
cache_lookup={str(r.page):Path(r.cache) for _,r in cache_status.iterrows()}
def contact(row,rank,kind):
 page=str(row.page);z=np.load(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);recon=z['geometry_t80'];mask=z['mask']>0;n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);cid=int(row.component);x,y,w,h=stats[cid,:4];m=45;x0=max(0,x-m);y0=max(0,y-m);x1=min(clean.shape[1],x+w+m);y1=min(clean.shape[0],y+h+m)
 panels=[clean[y0:y1,x0:x1],recon[y0:y1,x0:x1],(lab[y0:y1,x0:x1]==cid).astype(np.uint8)*255];titles=['clean truth','frozen geometry_t80','component mask']
 fig,axes=plt.subplots(1,3,figsize=(15,5))
 for ax,im,title in zip(axes,panels,titles):ax.imshow(im,cmap='gray',vmin=0,vmax=255);ax.set_title(title);ax.axis('off')
 fig.suptitle(f'{kind} #{rank} | {page} component {cid} | structural={row.structural_outcome} | cosmetic={row.cosmetic_outcome} | p_structural_safe={row.prob_structural_safe:.3f} | {row.structural_failure_reasons}')
 plt.tight_layout();plt.savefig(CONTACTS/f'{kind}_{rank:03d}_{page}_component_{cid:04d}.png',dpi=170,bbox_inches='tight');plt.close(fig)
categories={
 'false_safe':df[df.candidate_structural_auto_safe&~df.structural_safe].sort_values('prob_structural_safe',ascending=False),
 'missed_structural_safe':df[(~df.candidate_structural_auto_safe)&df.structural_safe].sort_values('prob_structural_safe'),
 'selected_cosmetic_residual':df[df.candidate_structural_auto_safe&~df.cosmetic_clean].sort_values('prob_structural_safe',ascending=False),
 'highest_structural_risk':df.sort_values('prob_structural_safe'),
}
for kind,data in categories.items():
 for rank,(_,row) in enumerate(data.head(MAX_CONTACTS).iterrows(),1):contact(row,rank,kind)


## Export predictions, manifest and decision summary


In [ ]:
pred_cols=['page','component','structural_safe','structural_outcome','structural_failure_reasons','cosmetic_clean','cosmetic_outcome','four_way_outcome','residual_annotation_fraction','prob_structural_safe','prob_cosmetic_clean','candidate_structural_auto_safe']
df[pred_cols].to_csv(TABLES/'out_of_fold_asymmetric_predictions.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'source_v7_5_run':str(V75_RUN),'reference_v7_6_run':str(V76_RUN) if V76_RUN else None,'frozen_candidate':'geometry_t80','primary_target':'structural_safe','secondary_target':'cosmetic_clean','clean_truth_role':'post-hoc labels and scoring only','page_grouped_cross_validation':True,'raw_coordinates_excluded':True,'components':int(len(df)),'pages':int(df.page.nunique()),'structural_safe':int(df.structural_safe.sum()),'structural_unsafe':int((~df.structural_safe).sum()),'cosmetic_clean':int(df.cosmetic_clean.sum()),'cosmetic_residual':int((~df.cosmetic_clean).sum()),'structural_gates':STRUCTURAL_GATES,'cosmetic_gates':COSMETIC_GATES,'predictors':predictors,'best_structural_model':best_struct,'best_cosmetic_model':best_cosmetic,'candidate_policy_is_production_authorisation':False,'diagnostic_threshold':t,'selected_components':int(df.candidate_structural_auto_safe.sum()),'selected_structural_failures':int((df.candidate_structural_auto_safe&~df.structural_safe).sum()),'independent_score_validation_required':True}
(OUT/'v7_7_manifest.json').write_text(json.dumps(manifest,indent=2))
summary=['V7.7 ASYMMETRIC STRUCTURAL-RISK CALIBRATION','',f'Frozen candidate: geometry_t80',f'Components: {len(df)} across {df.page.nunique()} pages',f'Structural-safe: {int(df.structural_safe.sum())}',f'Structural-unsafe: {int((~df.structural_safe).sum())}',f'Cosmetic-clean: {int(df.cosmetic_clean.sum())}',f'Cosmetic-residual: {int((~df.cosmetic_clean).sum())}',f'Best structural model: {best_struct}',f'Best cosmetic model: {best_cosmetic}',f'Diagnostic threshold: {t:.4f}',f'Candidate selected components: {int(df.candidate_structural_auto_safe.sum())}',f'Observed selected structural failures: {int((df.candidate_structural_auto_safe&~df.structural_safe).sum())}','','IMPORTANT: this is retrospective calibration only. Independent score/source-group validation is required before automatic correction.']
(OUT/'executive_summary.txt').write_text('\n'.join(summary));print('\n'.join(summary));print('V7.7 COMPLETE:',OUT)
